## Installation

### Prerequisites

```bash
pip install kaggle kagglehub pyyaml
```

---

### Kaggle Notebooks Environment

**Important:** When running code in **Kaggle Notebooks**, you don't need to install most packages manually!

Kaggle notebooks run on the [Kaggle Python Docker image](https://github.com/kaggle/docker-python), which comes with **hundreds of pre-installed packages** including:

- **Deep Learning:** `torch`, `tensorflow`, `keras`, `transformers`, `timm`
- **Data Science:** `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`, `seaborn`
- **Computer Vision:** `opencv-python`, `Pillow`, `albumentations`
- **NLP:** `nltk`, `spacy`, `gensim`
- **Utilities:** `tqdm`, `requests`, `beautifulsoup4`, `plotly`

**Full list:** See [kaggle_requirements.txt](https://github.com/Kaggle/docker-python/blob/main/kaggle_requirements.txt) for all pre-installed packages.

**What this means:**
- ✅ **No need to install** `torch`, `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`, etc.
- ✅ **Only install** packages that are **not** in the default image (e.g., `timm` if it's missing, or very new packages)
- ✅ **Check first:** Before running `!pip install`, verify if the package is already available

**Example:**
```python
# In Kaggle notebooks, these are already available:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score

# Only install if truly needed:
# !pip install -q some-new-package
```

---

### Kaggle Authentication

You need Kaggle API credentials to download datasets.

#### Option 1: Local Development (kaggle.json)

1. Go to [Kaggle Account Settings](https://www.kaggle.com/settings/account)
2. Scroll to "API" section
3. Click "Create New Token"
4. Download `kaggle.json`
5. Place it in the correct location:

**Linux/macOS:**
```bash
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json
```

**Windows:**
```powershell
# Place kaggle.json in:
C:\Users\<YourUsername>\.kaggle\kaggle.json
```
---

### Universal Kaggle-Compatible Docker Image

This repo also includes a **universal Docker image** configuration (see `Dockerfile` in the project root) that mirrors the Kaggle notebook environment, so you can run your experiments on RunPod (or locally) with almost exactly the same stack.

#### 1. Base Image

The image is built on Kaggle's official GPU image:

```dockerfile
FROM gcr.io/kaggle-gpu-images/python
```

This is the same family of images used by Kaggle Notebooks (`docker-python` repo: [kaggle/docker-python](https://github.com/kaggle/docker-python)), so you inherit all the pre-installed packages listed in
[`kaggle_requirements.txt`](https://github.com/Kaggle/docker-python/blob/main/kaggle_requirements.txt).

#### 2. Build the Universal Image (locally)

From the `src` directory (where `Dockerfile` lives):

```bash
cd /home/$USERNAME/anaconda3/envs/PyTorchTutorial/src

# Pick a generic, reusable name
docker build -t kaggle-universal:latest .
```

You can use this same image for **all Kaggle challenges** by mounting different project folders or using different entrypoints.

#### 3. Push to GitHub Container Registry (GHCR)

We'll use an environment variable `GITHUB_USERNAME` so the commands are reusable. By default, set it to your GitHub username **`behnamasadi`**:

```bash
export GITHUB_USERNAME=behnamasadi
```

1. **Login to GHCR** (you need a GitHub personal access token with `write:packages`):

```bash
echo $GITHUB_TOKEN | docker login ghcr.io -u $GITHUB_USERNAME --password-stdin
```

2. **Tag and push**:

```bash
# Tag local image for GHCR
docker tag kaggle-universal:latest ghcr.io/$GITHUB_USERNAME/kaggle-universal:latest

# Push to GHCR
docker push ghcr.io/$GITHUB_USERNAME/kaggle-universal:latest
```

You can now use `ghcr.io/$GITHUB_USERNAME/kaggle-universal:latest` as a **custom image** on RunPod (or any other GPU provider that supports custom Docker images).

#### 4. Push to Docker Hub (alternative)

If you prefer Docker Hub:

```bash
# Login interactively
docker login

# Tag and push
docker tag kaggle-universal:latest <dockerhub-username>/kaggle-universal:latest
docker push <dockerhub-username>/kaggle-universal:latest
```

#### 5. Using the Image on RunPod

On RunPod (or a similar service), configure your pod with:

- **Image**: `ghcr.io/<github-username>/kaggle-universal:latest` (or `<dockerhub-username>/kaggle-universal:latest`)
- **Mounts**: mount your local projects to `/workspace/host` if desired

Typical run command inside the container:

```bash
# Example: jump into your lung disease project
cd /workspace/projects/Lung_Disease_Dataset
python scripts/train.py --config configs/train_runpod.yaml
```

✅ This setup gives you a **single, reusable image** that:
- Matches Kaggle's Python stack closely
- Works across **all** your Kaggle challenges
- Avoids re-installing common ML packages in every notebook or container.


#### Option 2: RunPod/Docker (Environment Variables)

Set environment variables in your RunPod/Docker container:

```bash
export KAGGLE_USERNAME="your_username"
export KAGGLE_KEY="your_api_key"
```

Or add to your Dockerfile:
```dockerfile
ENV KAGGLE_USERNAME=your_username
ENV KAGGLE_KEY=your_api_key
```

### kaggle.json Locations Reference

| Platform | Location |
|----------|----------|
| **Linux** | `~/.kaggle/kaggle.json` or `~/.config/kaggle/kaggle.json` |
| **macOS** | `~/.kaggle/kaggle.json` |
| **Windows** | `C:\Users\<YourUsername>\.kaggle\kaggle.json` |
| **WSL** | `/home/<username>/.kaggle/kaggle.json` |

---

## Configuration

Edit `config/train.yaml`:

```yaml
data:
  kaggle_dataset: "owner/dataset-name"  # e.g., "masoudnickparvar/brain-tumor-mri-dataset"
  path: "./data/train"                  # Relative to kaggle_structure directory
```

---


## Finding Kaggle Datasets

### Method 1: Kaggle Website

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets)
2. Search for your topic with relevant keywords:
   - **Medical:**: `brain`, `tumor`, `MRI`, `liver`, `kidneys`, `heart`, `lungs`, `vessel`, `lesion` `COVID` `lung` `opacities`,  `breast` `cancer` `Ultrasound`, `histopathology`, `segmentation`, `medical imaging`
   - **Computer Vision:** `object detection`, `segmentation`, `saliency`, `image similarity`
   - **3D Vision:** `LiDAR`, `SLAM`, `Photogrammetry`, `stereo`, `monocular depth`
3. Click on a dataset
4. The dataset name is in the URL: `kaggle.com/datasets/OWNER/DATASET-NAME`
5. Use `OWNER/DATASET-NAME` in your `config/train.yaml`

### Method 2: Kaggle CLI

The Kaggle CLI is a powerful tool for searching and downloading datasets from the command line.


#### Searching Datasets

**Basic search:**
```bash
# Search by keyword
kaggle datasets list -s "MRI"
```

**Sort options:**

Available sort methods: `hottest`, `votes`, `updated`, `active`, `published`

```bash
# Sort by votes (most popular)
kaggle datasets list -s "MRI" --sort-by votes
```

**Pagination:**

Use `-p` or `--page` to see more results:

```bash
# Get page 3 with specific sorting
kaggle datasets list -s "medical imaging" --sort-by votes -p 3
```


```bash
# Show more results per page (max 100)
kaggle datasets list -s "segmentation" --max-size 50

```

Copy the dataset identifier (e.g., `masoudnickparvar/brain-tumor-mri-dataset`) and use it in your `config/train.yaml`.

---

#### Downloading Datasets

**Download directly with CLI:**

```bash
# Download a dataset
kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
```

### Method 3: Browse by Category

Visit specific categories:
- **Sports:** https://www.kaggle.com/datasets?topic=sports
- **Medicine:** https://www.kaggle.com/datasets?topic=health
- **Computer Vision:** https://www.kaggle.com/datasets?topic=computer-vision

---

## Kaggle Dataset Downloader
In your project, use this script, this create symlinks to your project data directory:

```python
import yaml
from utility.file_utils import resource_path
import pathlib
import os

try:
    import kagglehub
    KAGGLEHUB_AVAILABLE = True
except ImportError:
    KAGGLEHUB_AVAILABLE = False
    kagglehub = None


if not KAGGLEHUB_AVAILABLE:
    raise ImportError(
        "kagglehub is required for automatic dataset download. "
        "Install it with: pip install kagglehub"
    )
else:
    print(f"kagglehub is available: {kagglehub.__version__}")


config_path = resource_path("../config/train.yaml")

config_path_absolute = pathlib.Path(config_path).resolve()
if config_path_absolute.exists():
    print(f"Reading config from{config_path_absolute}")
else:
    print(f"No config found at {config_path_absolute}")
    exit()


with open(config_path_absolute) as f:
    cfg = yaml.load(f, Loader=yaml.SafeLoader)


print(cfg["data"]["kaggle_dataset"])
print(cfg["data"]["path"])


"""
Download the dataset from Kaggle if it doesn't already exist.
Checks if train and val directories exist and have content before downloading.

Uses KAGGLE_USERNAME and KAGGLE_KEY environment variables for authentication.
"""


try:
    print(f"\nAttempting to download dataset: {cfg['data']['kaggle_dataset']}")
    dataset_path = kagglehub.dataset_download(cfg["data"]["kaggle_dataset"])

    print(f"\n{'='*70}")
    print(f"✅ Dataset downloaded successfully")
    print(f"{'='*70}")
    print(f"  Source (Kaggle cache): {dataset_path}")

    # Create symlink from kaggle_structure data directory to Kaggle cache
    # Resolve path relative to kaggle_structure directory (parent of config directory)
    kaggle_structure_root = config_path_absolute.parent.parent
    target_path = (kaggle_structure_root / cfg["data"]["path"]).resolve()
    source_path = pathlib.Path(dataset_path)

    # Remove existing directory/symlink if it exists
    if target_path.exists() or target_path.is_symlink():
        if target_path.is_symlink():
            target_path.unlink()
        elif target_path.is_dir():
            print(f"  ⚠️  Target directory exists, skipping symlink creation")
            print(f"  Target (Project dir):  {target_path}")
            print(f"{'='*70}")
        else:
            target_path.unlink()

    if not target_path.exists():
        # Create parent directories if needed
        target_path.parent.mkdir(parents=True, exist_ok=True)

        # Create symlink
        target_path.symlink_to(source_path)
        print(f"  Target (Project dir):  {target_path}")
        print(f"{'='*70}")
        print(f"\n✅ Symlink created successfully")
        print(f"   {target_path} -> {source_path}")
        print(f"   (no disk space wasted)")
        print(f"{'='*70}")

except Exception as e:
    error_msg = (
        f"Failed to download or organize dataset: {e}\n"
        f"\nAuthentication options:\n"
        f"  - RunPod/Docker: Set KAGGLE_USERNAME and KAGGLE_KEY environment variables\n"
        f"  - Local: Use kagglehub.login() or create ~/.kaggle/kaggle.json\n"
        f"  - Or ensure dataset is already downloaded to: {cfg['data']['path']}"
    )
    raise RuntimeError(error_msg) from e
```

## What This Does

This script:
1. Reads dataset configuration from `config/train.yaml`
2. Downloads the specified Kaggle dataset using `kagglehub` (stores in cache: `~/.cache/kagglehub/`)
3. Creates a symlink from `kaggle_structure/data/train` to the downloaded dataset
4. Works both locally (using `kaggle.json`) and on RunPod (using environment variables)

---

## Running in Kaggle Notebooks

When running your code in a Kaggle notebook, the environment and paths are different from local/RunPod setups. Here's how to handle it properly.


### Environment Detection

Kaggle notebooks have specific paths and don't require authentication:

In [3]:
import os
import warnings
from tqdm import tqdm
import time
import kagglehub

warnings.filterwarnings('ignore')



# =============================================================================
# STEP 1: DATA DOWNLOAD
# =============================================================================
print("\n📥 STEP 1: Downloading dataset from KaggleHub...")
try:
    path = kagglehub.dataset_download("orvile/brain-cancer-mri-dataset")
    print("Path to dataset files:", path)
    data_path = path
except Exception as e:
    print(f"Error downloading dataset: {e}")
    print("Using local data path...")
    data_path = "data/brain-cancer/"

# Check if data exists
if not os.path.exists(data_path):
    print(f"Error: Data path '{data_path}' does not exist!")
    print("Please ensure the dataset is downloaded or available at the specified path.")


📥 STEP 1: Downloading dataset from KaggleHub...
Path to dataset files: /home/behnam/.cache/kagglehub/datasets/orvile/brain-cancer-mri-dataset/versions/2



## Kaggle Hardware & GPU

When running code in Kaggle notebooks or competitions, be aware of hardware constraints.

### Available GPUs

| GPU | VRAM | Compute Capability | Best For |
|-----|------|-------------------|----------|
| **NVIDIA Tesla P100** | 16 GB | 6.0 | General deep learning, FP32 |
| **NVIDIA Tesla T4** | 16 GB | 7.5 | Mixed precision (FP16), inference |

> **Note:** You **cannot choose** which GPU you get. Kaggle assigns P100 or T4 randomly based on availability. Both have **16GB VRAM** - design your code for this limit.

### Checking Which GPU You Got

```python
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_props.total_memory / 1e9:.1f} GB")
    print(f"Compute Capability: {gpu_props.major}.{gpu_props.minor}")
    
    if "P100" in gpu_name:
        print("→ Good for FP32 training")
    elif "T4" in gpu_name:
        print("→ Excellent for FP16/mixed precision (3-4x faster)")
else:
    print("No GPU available")
```


### GPU Comparison

| Aspect | P100 | T4 |
|--------|------|-----|
| **FP32 performance** | 9.3 TFLOPS (better) | 8.1 TFLOPS |
| **FP16 performance** | 18.7 TFLOPS | 65 TFLOPS (3-4x faster!) |
| **Memory bandwidth** | 732 GB/s (better) | 320 GB/s |
| **Power** | 250W | 70W (more efficient) |

**Bottom line:** Use mixed precision (FP16) - T4 will be much faster, P100 still works fine.

### System Resources

| Resource | Limit |
|----------|-------|
| **GPU Time** | ~30 hours/week |
| **CPU Cores** | 4 cores |
| **RAM** | 16-30 GB |
| **Disk Space** | ~73 GB (temporary) |
| **Session Time** | 9-12 hours max |
| **Internet** | Enabled (disabled in competitions) |

### Critical Limitations

1. 16 GB VRAM (Most Common Bottleneck)
2. Session Timeout (9-12 hours)
3. Disk Space (~73 GB)



---

## GPU Optimization Best Practices

Strategies to maximize GPU utilization and avoid out-of-memory errors on Kaggle's 16GB GPUs.

1. Use Mixed Precision Training (FP16)
2. Gradient Accumulation
3. Monitor GPU Memory
4. Find Optimal Batch Size Dynamically
5. Clear Unused Variables & Cache
6. Gradient Checkpointing
7. Use Efficient Data Loading
8. Reduce Model Precision Strategically
9. GPU-Agnostic Code Pattern

## Uploading and Creating your Own Dataset
[rvp group slam dataset](https://rvp-group.net/slam-dataset.html)

Refs [1](https://www.kaggle.com/docs/api#installation)